<a href="https://colab.research.google.com/github/jusceliadesouza/button-and-cursors/blob/main/aula_titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚢 O Naufrágio dos Dados: Por que o Titanic afunda de novo sem limpeza?

**Bem-vindos a bordo!** Nossa missão hoje é investigar o naufrágio mais famoso do mundo. Mas temos um problema real: dados do mundo real são bagunçados. Se alimentarmos um modelo preditivo com a base do jeito que ela vem, teremos o famoso efeito *Garbage In, Garbage Out* (Lixo entra, Lixo sai).

Hoje, serei a guia de vocês para transformar o caos em informações confiáveis.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno

# Configuração visual para os gráficos ficarem bonitos no projetor/tela
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context("talk")

# 1. Carregando os dados
titanic_raw = sns.load_dataset('titanic')
titanic_clean = titanic_raw.copy()

# 2. A primeira olhada: O que temos aqui?
display(titanic_raw.head())

# 3. O Resumo do Caos
print("\n--- Informações da Base ---")
titanic_raw.info()

### ⌨️ O Problema da Digitação Livre: Quantos gêneros existem na base?

Até agora resolvemos dados que faltavam e dados extremos. Mas e quando o dado está lá, só que foi digitado de qualquer jeito? Imagine que o formulário de embarque do Titanic era de texto livre. Alguém escreveu 'Female', o outro 'F', o outro 'Fem' com um espaço no final. Para nós, humanos, é a mesma coisa. Para a máquina, são categorias completamente diferentes. Vamos simular esse caos e ver o que acontece.

In [ ]:
titanic_clean.loc[0:15, 'sex'] = 'F'
titanic_clean.loc[16:30, 'sex'] = 'Fem '      # Note o espaço no final!
titanic_clean.loc[31:45, 'sex'] = 'FEMALE'
titanic_clean.loc[46:60, 'sex'] = 'm'
titanic_clean.loc[61:75, 'sex'] = 'MALE '     # Espaço no final também

# O pesadelo do analista: Tentando fazer um gráfico simples de contagem
plt.figure(figsize=(10, 5))
sns.countplot(data=titanic_clean, x='sex', palette='Set2')
plt.title("O Caos Inconsistente: Como o computador enxerga os dados", fontsize=16)
plt.xticks(rotation=45)
plt.show()

# Vendo os números exatos
print("Contagem de Categorias (O computador acha que existem 7 gêneros!):")
print(titanic_clean['sex'].value_counts())

### 📝 A Padronização de Texto

Viram o estrago? Se eu colocar isso em um modelo de Machine Learning, ele vai achar que 'Fem ' (com espaço) tem uma taxa de sobrevivência diferente de 'FEMALE' (maiúsculo). A regra de ouro da padronização de texto tem 3 passos: colocar tudo em minúsculo (ou maiúsculo), arrancar espaços sobrando nas pontas, e unificar as siglas.

In [ ]:
# PASSO 1: Colocar tudo em letras minúsculas (lower case)
titanic_clean['sex'] = titanic_clean['sex'].str.lower()

# PASSO 2: Remover espaços em branco perdidos no início e no fim (strip)
# Isso resolve o problema invisível do 'fem ' vs 'fem'
titanic_clean['sex'] = titanic_clean['sex'].str.strip()

# PASSO 3: Mapear e substituir as variações para um padrão único (replace)
dicionario_correcao = {
    'f': 'female',
    'fem': 'female',
    'm': 'male'
}
titanic_clean['sex'] = titanic_clean['sex'].replace(dicionario_correcao)

# Verificando a paz restaurada
plt.figure(figsize=(6, 4))
sns.countplot(data=titanic_clean, x='sex', palette='Set2')
plt.title("Paz Restaurada: Dados Padronizados", fontsize=16)
plt.show()

print("Contagem após a limpeza:")
print(titanic_clean['sex'].value_counts())

### 🤖 O Teste de Fogo: Colocando a Inteligência Artificial à Prova

Chegou a hora da verdade. Nós construímos duas realidades: o titanic_sujo (onde preenchemos os dados faltantes de qualquer jeito só para o algoritmo não dar erro e mantivemos a bagunça de digitação) e o titanic_limpo (onde tratamos tudo como verdadeiros detetives). Vamos treinar o mesmo algoritmo de Machine Learning (um Random Forest) nas duas bases e ver quem prevê melhor quem sobrevive e quem não.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# 1. PREPARANDO A BASE SUJA (O jeito "Apressado")
# ==========================================
# Simulando a sujeira que criamos antes
titanic_sujo = sns.load_dataset('titanic')
titanic_sujo.loc[0:15, 'sex'] = 'F'
titanic_sujo.loc[16:30, 'sex'] = 'Fem '
titanic_sujo.loc[31:45, 'sex'] = 'FEMALE'

# O scikit-learn não aceita Nulos. O analista apressado preenche tudo com -1!
titanic_sujo = titanic_sujo[['survived', 'pclass', 'sex', 'age', 'fare']].fillna(-1)

# Transformando texto em colunas (O modelo vai criar uma coluna para 'F', outra para 'Fem ', etc)
titanic_sujo = pd.get_dummies(titanic_sujo, columns=['sex'])

# Separando Treino e Teste
X_sujo = titanic_sujo.drop('survived', axis=1)
y_sujo = titanic_sujo['survived']
X_train_sujo, X_test_sujo, y_train_sujo, y_test_sujo = train_test_split(X_sujo, y_sujo, test_size=0.3, random_state=42)

# ==========================================
# 2. PREPARANDO A BASE LIMPA (O jeito "Detetive")
# ==========================================
# Usando as colunas numéricas que já limpamos lá em cima!
features_limpas = ['pclass', 'age', 'fare', 'sex_encoded', 'family_size']
X_limpo = titanic_clean[features_limpas]
y_limpo = titanic_clean['survived'] # Alvo

X_train_limpo, X_test_limpo, y_train_limpo, y_test_limpo = train_test_split(X_limpo, y_limpo, test_size=0.3, random_state=42)

Vamos usar um modelo de Floresta Aleatória (Random Forest). É como se juntássemos várias árvores de decisão e pedíssemos para elas votarem se um passageiro sobrevive ou não com base nos dados. Vamos treinar os dois e comparar as acurácias (taxa de acerto).

In [ ]:
# Treinando o Modelo Sujo
modelo_sujo = RandomForestClassifier(random_state=42, n_estimators=100)
modelo_sujo.fit(X_train_sujo, y_train_sujo)
previsoes_sujo = modelo_sujo.predict(X_test_sujo)
acuracia_suja = accuracy_score(y_test_sujo, previsoes_sujo)

# Treinando o Modelo Limpo
modelo_limpo = RandomForestClassifier(random_state=42, n_estimators=100)
modelo_limpo.fit(X_train_limpo, y_train_limpo)
previsoes_limpo = modelo_limpo.predict(X_test_limpo)
acuracia_limpa = accuracy_score(y_test_limpo, previsoes_limpo)

# Visualizando o Impacto
resultados = pd.DataFrame({
    'Cenário': ['Base Suja & Apressada', 'Base Limpa & Trabalhada'],
    'Acurácia (Taxa de Acerto)': [acuracia_suja, acuracia_limpa]
})

plt.figure(figsize=(8, 5))
ax = sns.barplot(x='Cenário', y='Acurácia (Taxa de Acerto)', data=resultados, palette=['salmon', 'lightgreen'])
plt.title("O Veredito do Machine Learning", fontsize=16)
plt.ylim(0.60, 0.90) # Ajustando o eixo Y para dar zoom na diferença

# Adicionando os números no topo das barras
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1%}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 10), textcoords='offset points', fontsize=14, fontweight='bold')

plt.show()

print("\n--- Conclusão Técnica ---")
print("O modelo limpo não apenas acerta mais, como é mais confiável.")
print("A base suja forçou o modelo a aprender padrões falsos (como idades negativas ou categorias duplicadas de gênero).")

Vocês viram que a Floresta Aleatória quase empatou  com os dados sujos. Isso é uma armadilha! Algoritmos de árvore às vezes decoram o lixo e nos enganam (overfitting). Mas o que acontece se usarmos um modelo matemático estrito, que depende de proporções reais, como a Regressão Logística? Um passageiro com idade '-1' ou uma tarifa distorcida vai destruir a matemática do modelo. Vamos ver!

In [ ]:
from sklearn.linear_model import LogisticRegression

# Treinando a Regressão Logística Suja
# max_iter=1000 para tentar forçar o modelo matemático a entender a bagunça
modelo_lr_sujo = LogisticRegression(max_iter=1000, random_state=42)
modelo_lr_sujo.fit(X_train_sujo, y_train_sujo)
previsoes_lr_sujo = modelo_lr_sujo.predict(X_test_sujo)
acuracia_lr_suja = accuracy_score(y_test_sujo, previsoes_lr_sujo)

# Treinando a Regressão Logística Limpa
modelo_lr_limpo = LogisticRegression(max_iter=1000, random_state=42)
modelo_lr_limpo.fit(X_train_limpo, y_train_limpo)
previsoes_lr_limpo = modelo_lr_limpo.predict(X_test_limpo)
acuracia_lr_limpa = accuracy_score(y_test_limpo, previsoes_lr_limpo)

# Visualizando o impacto devastador
resultados_lr = pd.DataFrame({
    'Cenário': ['Base Suja', 'Base Limpa'],
    'Acurácia': [acuracia_lr_suja, acuracia_lr_limpa]
})

plt.figure(figsize=(8, 5))
ax = sns.barplot(x='Cenário', y='Acurácia', data=resultados_lr, palette=['darkred', 'darkgreen'])
plt.title("Regressão Logística: Quando a matemática não perdoa a sujeira", fontsize=16)

# Destacando o eixo Y para evidenciar a queda
plt.ylim(0.50, 0.90)

# Adicionando os números no topo
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1%}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 10), textcoords='offset points', fontsize=14, fontweight='bold')

plt.show()

print("\n--- Conclusão do Detetive de Dados ---")
print("Olhem a diferença! A Regressão Logística despenca com dados sujos.")
print("A lição de hoje: Nunca confie cegamente no algoritmo. O único jeito de garantir um modelo robusto é limpando a base antes!")

### 🔍 O Raio-X dos Dados Faltantes

O comando `info()` já nos deu uma pista de que faltam dados na coluna de idade (`age`) e convés (`deck`). Mas números em uma tabela são difíceis de digerir. Vamos usar a biblioteca `missingno` para literalmente tirar um Raio-X da nossa base.

In [ ]:
# Gráfico de matriz de nulidade
msno.matrix(titanic_raw, figsize=(12, 6), color=(0.2, 0.4, 0.6))
plt.title("Matriz de Dados Faltantes: As lacunas da história", fontsize=20, pad=20)
plt.show()

# Gráfico de barras para quantificar
msno.bar(titanic_raw, figsize=(12, 4), color=(0.8, 0.4, 0.4))
plt.title("Proporção de Dados Preenchidos", fontsize=16)
plt.show()

### 🛟 Operação de Resgate (Tratando Nulos)

Temos três cenários de nulos aqui, e para cada um, uma técnica diferente:
1. **Falta DEMAIS:** A coluna `deck` tem quase 80% de nulos. Tentar adivinhar seria invenção pura. Vamos descartar (Drop).
2. **Falta POUCO:** A coluna `embark_town` tem 2 nulos. Podemos preencher com a cidade mais comum (Moda).
3. **O Desafio Intermediário:** A coluna `age` tem 20% de nulos. A idade é crucial para sobreviver (lembrem: 'mulheres e crianças primeiro'). Não podemos descartar, mas a média geral emburrece o dado. Vamos imputar com inteligência.

In [ ]:
# 1. Drop: Removendo o que não tem salvação
titanic_clean.drop(columns=['deck'], inplace=True)

# 2. Moda: Preenchendo lacunas minúsculas
moda_embarque = titanic_clean['embark_town'].mode()[0]
titanic_clean['embark_town'] = titanic_clean['embark_town'].fillna(moda_embarque)

# 3. Imputação Condicional (Intermediário)
# Vamos ver a diferença de idade por Classe e Gênero antes de imputar
plt.figure(figsize=(10, 5))
sns.boxplot(data=titanic_clean, x='pclass', y='age', hue='sex')
plt.title("A idade varia muito dependendo da Classe e Gênero!")
plt.show()

# A mágica: preenchendo a idade baseada na mediana do grupo específico do passageiro
titanic_clean['age'] = titanic_clean.groupby(['pclass', 'sex'])['age'].transform(lambda x: x.fillna(x.median()))

print("\nVerificação pós-limpeza de nulos:")
print(titanic_clean[['age', 'embark_town']].isnull().sum())

### 💰 Caçando Anomalias e Outliers

A base não tem mais buracos, mas será que faz sentido? Vamos olhar a tarifa (`fare`). Em modelos baseados em regressão, valores extremos puxam a linha de tendência e distorcem os pesos. Vamos ver se temos bilhetes milionários distorcendo a realidade.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# Antes do tratamento
sns.boxplot(x=titanic_clean['fare'], ax=ax[0], color='salmon')
ax[0].set_title('Antes: Passageiros que pagaram +$500')

# Tratamento: Técnica de Winsorize (Clipping) no percentil 95
limite_superior = titanic_clean['fare'].quantile(0.95)
titanic_clean['fare'] = np.where(titanic_clean['fare'] > limite_superior, limite_superior, titanic_clean['fare'])

# Depois do tratamento
sns.boxplot(x=titanic_clean['fare'], ax=ax[1], color='lightgreen')
ax[1].set_title('Depois: Valores extremos limitados')

plt.tight_layout()
plt.show()

### 🤖 Traduzindo para a Máquina (Feature Engineering)

Para fechar nossa faxina com chave de ouro: algoritmos de Machine Learning geralmente não entendem texto (como 'male' ou 'female'). Precisamos converter variáveis categóricas em números. Isso se chama *Encoding*. Além disso, vamos criar uma nova variável que consolida a informação familiar.

In [ ]:
# 1. Feature Engineering: Tamanho da família (irmãos/cônjuges + pais/filhos + o próprio passageiro)
titanic_clean['family_size'] = titanic_clean['sibsp'] + titanic_clean['parch'] + 1
titanic_raw['family_size'] = titanic_raw['sibsp'] + titanic_raw['parch'] + 1

# 2. Encoding: Transformando texto em número
titanic_clean['sex_encoded'] = titanic_clean['sex'].map({'male': 0, 'female': 1})

# Transformando cidade de embarque em colunas dummy (One-Hot Encoding)
titanic_clean = pd.get_dummies(titanic_clean, columns=['embark_town'], drop_first=True, dtype=int)

display(titanic_clean[['sex', 'sex_encoded', 'family_size']].head())

### ♦️ O Grande Final (O Custo da Sujeira)

Chegamos ao ápice da aula. Vamos provar por que passamos os últimos minutos limpando. Vamos fazer uma análise e ver como os dados sujos escondiam os verdadeiros padrões de sobrevivência.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Criando faixas etárias para facilitar a visualização
bins = [0, 12, 18, 35, 60, 100]
labels = ['Criança', 'Adolescente', 'Jovem Adulto', 'Adulto', 'Idoso']

# No dado sujo, quem não tem idade (nulo) desaparece dessa análise!
titanic_raw['age_group'] = pd.cut(titanic_raw['age'], bins=bins, labels=labels)
titanic_clean['age_group'] = pd.cut(titanic_clean['age'], bins=bins, labels=labels)

sns.barplot(data=titanic_raw, x='age_group', y='survived', ax=ax[0], palette='Reds', errorbar=None)
ax[0].set_title('Análise com Dados Sujos\n(Ignora 177 passageiros sem idade)', fontsize=14)
ax[0].set_ylabel('Taxa de Sobrevivência')

sns.barplot(data=titanic_clean, x='age_group', y='survived', ax=ax[1], palette='Greens', errorbar=None)
ax[1].set_title('Análise com Dados Limpos\n(Idades imputadas com inteligência)', fontsize=14)
ax[1].set_ylabel('Taxa de Sobrevivência')

plt.tight_layout()
plt.show()

print("\nConclusão Visível:")
print("Com os dados limpos, a taxa de sobrevivência das crianças fica muito mais precisa e estabilizada.")
print("Sem a limpeza, o seu modelo teria menos exemplos para aprender padrões vitais!")